In [5]:
from rich import print
from sqlmodel import Session, create_engine, select

from cali.runner import CaliRunner
from cali.sqlmodel import AnalysisSettings, CaliResult, Experiment
from cali.sqlmodel._model import DetectionSettings

In [ ]:
def table(db_path: str) -> None:
    engine = create_engine(f"sqlite:///{db_path}")
    with Session(engine) as session:
        # Get all AnalysisResults
        results = session.exec(select(CaliResult).order_by(CaliResult.id)).all()

        print("=" * 100)
        print(
            f"{'ID':<5} {'Created At':<20} {'Experiment ID':<15} {'Detection ID':<15} {'Analysis ID':<15} {'Positions':<15}"
        )
        print("-" * 100)

        for result in results:
            created_at = result.created_at.strftime("%Y-%m-%d %H:%M:%S")
            experiment_id = str(result.experiment) if result.experiment else "None"
            detection_id = (
                str(result.detection_settings) if result.detection_settings else "None"
            )
            analysis_id = (
                str(result.analysis_settings) if result.analysis_settings else "None"
            )
            positions = (
                str(result.positions_analyzed) if result.positions_analyzed else "None"
            )

            print(
                f"{result.id:<5} {created_at:<20} {experiment_id:<15} {detection_id:<15} {analysis_id:<15} {positions:<15}"
            )

In [9]:
data_path = "/Volumes/T7 Shield/for FG/TSC_hSynLAM77_ACTX250730_D36/TSC_hSynLAM77_ACTX250730_D36_DIV54_250923_jRCaMP1b_Spt.tensorstore.zarr"

In [10]:
cali = CaliRunner()

In [12]:
experiment = Experiment.create_from_data(
    name="New Experiment",
    data_path=data_path,
    plate_maps={
        "genotype": {"B5": "WT"},
        "treatment": {"B5": "Vehicle"},
    },
)
print(experiment)
print(experiment.plate)

Experiment(
    name='New Experiment',
    description=None,
    experiment_type='Spontaneous Activity',
    id=None,
    created_at=datetime.datetime(2025, 11, 22, 21, 37, 20, 258248)
)

Plate(
    name='96-well',
    plate_type='96-well',
    rows=8,
    columns=12,
    plate_maps={'genotype': {'B5': 'WT'}, 'treatment': {'B5': 'Vehicle'}},
    id=None
)

In [13]:
cali.run(
    experiment,
    data_path,
    detection_settings=DetectionSettings(method="cellpose", model_type="cpsam"),
    analysis_settings=AnalysisSettings(dff_window=150),
    global_position_indices=[0],
    overwrite=True,
)

2025-11-22 21:37:38,767 - cali_logger - INFO - 🔄 Overwriting existing database at /Volumes/T7 Shield/for FG/TSC_hSynLAM77_ACTX250730_D36/results.cali
2025-11-22 21:37:39,021 - cali_logger - INFO - 💾 Experiment analysis updated and saved to database at /Volumes/T7 Shield/for FG/TSC_hSynLAM77_ACTX250730_D36/results.cali.
2025-11-22 21:37:39,027 - cali_logger - INFO - ⚙️ Created new DetectionSettings ID 1 (method: cellpose)
2025-11-22 21:37:39,032 - cali_logger - INFO - ⚙️ Created new AnalysisSettings ID 1
2025-11-22 21:37:39,033 - cali_logger - INFO - 🔍 Running detection on 1 positions...
2025-11-22 21:37:41,087 - cali_logger - INFO - Use GPU: True
2025-11-22 21:37:41,087 - cali_logger - INFO - Loading model from `cpsam`.
2025-11-22 21:37:42,819 - cali_logger - INFO - Loading images for batch processing...
2025-11-22 21:37:48,088 - cali_logger - INFO - Processing 1 images in batches of 8
Running Cellpose: 100%|██████████| 1/1 [00:13<00:00, 13.01s/it]
2025-11-22 21:38:01,618 - cali_logger

In [20]:
table(cali.database_path)

====================================================================================================

ID    Created At           Experiment ID   Detection ID    Analysis ID     Positions

----------------------------------------------------------------------------------------------------

1     2025-11-22 21:38:17  1               1               1               [0]

In [21]:
cali.run(
    experiment,
    data_path,
    detection_settings=DetectionSettings(method="cellpose", model_type="cpsam"),
    analysis_settings=AnalysisSettings(dff_window=150),
    global_position_indices=[0, 1],
)

2025-11-22 21:39:40,472 - cali_logger - INFO - ♻️ Reusing existing DetectionSettings ID 1 (method: cellpose)
2025-11-22 21:39:40,475 - cali_logger - INFO - ♻️ Reusing existing AnalysisSettings ID 1
2025-11-22 21:39:40,476 - cali_logger - INFO - ⚠️  Detection exists for 1 position(s) but missing for 1 position(s): [1]. Running detection for missing positions.
2025-11-22 21:39:40,476 - cali_logger - INFO - 🔍 Running detection on 2 positions...
2025-11-22 21:39:40,478 - cali_logger - INFO - Use GPU: True
2025-11-22 21:39:40,478 - cali_logger - INFO - Loading model from `cpsam`.
2025-11-22 21:39:42,508 - cali_logger - INFO - Loading images for batch processing...
2025-11-22 21:39:55,413 - cali_logger - INFO - Processing 2 images in batches of 8
Running Cellpose: 100%|██████████| 1/1 [00:27<00:00, 27.69s/it]
2025-11-22 21:40:25,806 - cali_logger - INFO - ✅ Detection complete: 407 ROIs detected across 2 FOVs
2025-11-22 21:40:25,808 - cali_logger - INFO - ✅ Detection committed: 407 ROIs acros

In [22]:
table(cali.database_path)

====================================================================================================

ID    Created At           Experiment ID   Detection ID    Analysis ID     Positions

----------------------------------------------------------------------------------------------------

1     2025-11-22 21:38:17  1               1               1               [0]

2     2025-11-22 21:40:47  1               1               1               [0]